## Week 36
Familiarize yourself with the dataset card, download the dataset and explore its
columns. Summarize data statistics (size, word count, etc.) for training and
validation data in the languages Arabic (ar), Korean (ko) and Telugu (te).

For each of the languages Arabic, Korean and Telugu, report the 5 most
common words in the questions from the training set and their count, as well
as their English translation. What kind of words are they?

Implement a rule-based classifier that predicts whether a question is an-
swerable or impossible, only using the document (context) and question.

You
may use machine translation as a component. Use the answerable field to
evaluate it on the validation set. What is the performance of your classifier for
each of the languages Arabic, Korean and Telugu?

In [ ]:
from datasets import load_dataset
dataset = load_dataset("coastalcph/tydi_xor_rc")
train_set = dataset["train"]
validation_set = dataset["validation"]

In [1]:
import pandas as pd
import string
import numpy as np

In [2]:
validation_set = pd.read_parquet("data/validation.parquet")
train_set = pd.read_parquet("data/train.parquet")

In [3]:
strip_punctuation = lambda line: [word.strip(string.punctuation+"؟")for word in line.split(" ")]
count_words = lambda line: len(line)

In [4]:
#!pip install torch

In [5]:
from transformers import pipeline
import torch

In [6]:
import torch
print("Torch:", torch.__version__)        # should show +cu126
print("CUDA runtime:", torch.version.cuda) # '12.6'
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.8.0+cu126
CUDA runtime: 12.6
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [7]:
from datasets import Dataset
from transformers import pipeline
import pandas as pd
import torch
from itertools import chain


In [8]:
LANG_CODE = {"ko": "kor_Hang", "ar": "arb_Arab", "te": "tel_Telu"}
device = 0 if torch.cuda.is_available() else -1

translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    tokenizer="facebook/nllb-200-distilled-600M",
    device=device
)



Device set to use cuda:0


In [9]:
def hf_translate_questions(lang_df, l, batch_size: int = 32):
    work_df = lang_df[["question"]].copy()
    work_df["question"] = work_df["question"].fillna("").astype(str)

    ds = Dataset.from_pandas(work_df.reset_index(drop=True), preserve_index=False)

    def _translate_batch(batch):
        outs = translator(
            batch["question"],
            src_lang=LANG_CODE[l],
            tgt_lang="eng_Latn",
            max_length=256,
            truncation=True,
        )
        return {"question_translated": [o["translation_text"] for o in outs]}

    ds_out = ds.map(_translate_batch, batched=True, batch_size=batch_size)

    translated_df = ds_out.to_pandas()
    return translated_df[["question_translated"]]



In [32]:
def safe_top_tokens(lang_df, l, topn: int = 5):
    col = "question_stripped"
    if lang_df[col].dtype == object and lang_df[col].map(lambda x: isinstance(x, list)).any():
        tokens = list(chain.from_iterable(x or [] for x in lang_df[col].tolist()))
    else:
        tokens = list(chain.from_iterable((str(x or "")).split() for x in lang_df[col].tolist()))
    tokens = [t.strip() for t in tokens if isinstance(t, str) and t.strip()]
    counts = pd.Series(tokens).value_counts().head(topn)

    def translate_top_words(counts_series, lcode):
        tx = [
            translator(w, src_lang=LANG_CODE[lcode], tgt_lang="eng_Latn", max_length=64)[0]["translation_text"]
            for w in counts_series.index
        ]
        out = counts_series.reset_index()
        out.columns = ["token", "count"]
        out["translation"] = tx
        return out

    return translate_top_words(counts, l)


In [22]:

def statistics(df, col=["question", "context"], translate = False):
    langs = ["ko", "ar", "te"]
    df = df[df["lang"].isin(langs)].copy()

    for c in col:
        df[f"{c}_stripped"] = df[c].apply(strip_punctuation)
        df[f"{c}_wordcount"] = df[f"{c}_stripped"].apply(count_words)

    print(
        df.groupby(["lang", "answerable"]).agg(
            question_wordcount_mean=("question_wordcount", "mean"),
            context_wordcount_mean=("context_wordcount", "mean"),
            question_wordcount_sum=("question_wordcount", "sum"),
            context_wordcount_sum=("context_wordcount", "sum"),
            question_wordcount_max=("question_wordcount", "max"),
            question_wordcount_min=("question_wordcount", "min"),
            count=("question_wordcount", "count")
        ).round(0)
    )

    df["question_translated"] = None

    for l in langs:
        mask = df["lang"] == l
        lang_df = df.loc[mask].copy()

        print(f"\nTop tokens for {l} -> en:")
        top_counts = safe_top_tokens(lang_df, l)
        print(top_counts)

        if translate:
            translated_df = hf_translate_questions(lang_df, l, batch_size=32)
            df.loc[mask, "question_translated"] = translated_df["question_translated"].values
            torch.cuda.empty_cache()

    return df

In [23]:
translate = False

In [33]:
val_for_stat = statistics(validation_set, translate=translate)


                 question_wordcount_mean  context_wordcount_mean  \
lang answerable                                                    
ar   False                           8.0                   112.0   
     True                            7.0                   103.0   
ko   False                           5.0                   108.0   
     True                            5.0                    95.0   
te   False                           6.0                   112.0   
     True                            6.0                   105.0   

                 question_wordcount_sum  context_wordcount_sum  \
lang answerable                                                  
ar   False                          406                   5813   
     True                          2436                  37285   
ko   False                           95                   2051   
     True                          1636                  32124   
te   False                          575                  10

In [31]:
train_for_stat = statistics(train_set, translate=translate)

                 question_wordcount_mean  context_wordcount_mean  \
lang answerable                                                    
ar   False                           8.0                   124.0   
     True                            7.0                   102.0   
ko   False                           5.0                   104.0   
     True                            5.0                    97.0   
te   False                           6.0                   118.0   
     True                            6.0                    87.0   

                 question_wordcount_sum  context_wordcount_sum  \
lang answerable                                                  
ar   False                         1948                  31580   
     True                         15509                 234189   
ko   False                          325                   6574   
     True                         11524                 228653   
te   False                          277                   5

In [ ]:
if translate:
    val_for_stat.to_csv("data/validation_translated.csv")
    train_for_stat.to_csv("data/train_translated.csv")

  I wanted to strip punctuation, but certain languages that we do not include in the analysis have special punctuation that has to be included in thw stripping pool

In [34]:
val_tr = pd.read_csv("data/validation_translated.csv")
train_tr = pd.read_csv("data/train_translated.csv")

In [35]:
train_tr

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"['30년', '전쟁의', '승자는', '누구인가']",4,"['The', 'conflict', 'between', 'France', 'and'...",108,Who is the winner of the Thirty Years' War?
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"['엑스선은', '누가', '발견하였는가']",3,"['X-rays', 'make', 'up', 'X-radiation', 'a', '...",122,Who discovered X-rays?
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"['아테네에서', '언제', '가장', '최근의', '올림픽이', '올렸나요']",6,"['In', '2022', 'Beijing', 'will', 'become', 't...",197,When was the last Olympic Games held in Athens?
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"['세상에서', '가장', '오래된', '방송사는', '무엇인가']",5,"['The', 'British', 'Broadcasting', 'Corporatio...",70,What's the oldest broadcaster in the world?
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"['팔레스타인', '수도는', '어딘가요']",3,"['Palestine', '', '', 'officially', 'the', 'St...",84,Where's the Palestinian capital?
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6330,15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,NaN,"['소말리아는', '2차', '개헌을', '언제', '했나요']",5,"['In', 'February', '2012', 'Somali', 'governme...",181,When did Somalia make its second inauguration?
6331,15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,NaN,"['세상에서', '가장', '먼저', '시작된', '교통수단은', '무엇인가']",6,"['The', 'first', 'earth', 'tracks', 'were', 'c...",118,What was the world's first transportation system?
6332,15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,NaN,"['2019년', '이집트의', '지도자는', '누구인가']",4,"['Abdel', 'Fattah', 'Saeed', 'Hussein', 'Khali...",30,Who is Egypt's leader in 2019?
6333,15341,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,NaN,"['독일에서', '가장', '인구밀도가', '높은', '도시는', '무엇인가']",6,"['Munich', '', '', '', 'is', 'the', 'capital',...",118,What is the most densely populated city in Ger...


In [36]:
statistics(train_tr, col = ["question", "context", "question_translated"])

                 question_wordcount_mean  context_wordcount_mean  \
lang answerable                                                    
ar   False                           8.0                   124.0   
     True                            7.0                   102.0   
ko   False                           5.0                   104.0   
     True                            5.0                    97.0   
te   False                           6.0                   118.0   
     True                            6.0                    87.0   

                 question_wordcount_sum  context_wordcount_sum  \
lang answerable                                                  
ar   False                         1948                  31580   
     True                         15509                 234189   
ko   False                          325                   6574   
     True                         11524                 228653   
te   False                          277                   5

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated,question_translated_stripped,question_translated_wordcount
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"[30년, 전쟁의, 승자는, 누구인가]",4,"[The, conflict, between, France, and, Spain, c...",108,None,"[Who, is, the, winner, of, the, Thirty, Years,...",9
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"[엑스선은, 누가, 발견하였는가]",3,"[X-rays, make, up, X-radiation, a, form, of, e...",122,None,"[Who, discovered, X-rays]",3
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"[아테네에서, 언제, 가장, 최근의, 올림픽이, 올렸나요]",6,"[In, 2022, Beijing, will, become, the, first-e...",197,None,"[When, was, the, last, Olympic, Games, held, i...",9
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"[세상에서, 가장, 오래된, 방송사는, 무엇인가]",5,"[The, British, Broadcasting, Corporation, BBC,...",70,None,"[What's, the, oldest, broadcaster, in, the, wo...",7
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"[팔레스타인, 수도는, 어딘가요]",3,"[Palestine, , , officially, the, State, of, Pa...",84,None,"[Where's, the, Palestinian, capital]",4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6330,15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,NaN,"[소말리아는, 2차, 개헌을, 언제, 했나요]",5,"[In, February, 2012, Somali, government, offic...",181,None,"[When, did, Somalia, make, its, second, inaugu...",7
6331,15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,NaN,"[세상에서, 가장, 먼저, 시작된, 교통수단은, 무엇인가]",6,"[The, first, earth, tracks, were, created, by,...",118,None,"[What, was, the, world's, first, transportatio...",7
6332,15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,NaN,"[2019년, 이집트의, 지도자는, 누구인가]",4,"[Abdel, Fattah, Saeed, Hussein, Khalil, El-Sis...",30,None,"[Who, is, Egypt's, leader, in, 2019]",6
6333,15341,독일에서 가장 인구밀도가 높은 도시는 무엇인가?,Munich (; ; ) is the capital and most populous...,ko,True,205,Berlin,NaN,"[독일에서, 가장, 인구밀도가, 높은, 도시는, 무엇인가]",6,"[Munich, , , , is, the, capital, and, most, po...",118,None,"[What, is, the, most, densely, populated, city...",9


In [37]:
def count_translated_quest_tokens(df, topn: int = 5): #irrelevant
    langs = ["ko", "ar", "te"]
    col  = "question_translated"
    for l in langs:
        mask = df["lang"] == l
        lang_df = df.loc[mask].copy()
        if lang_df[col].dtype == object and lang_df[col].map(lambda x: isinstance(x, list)).any():
            tokens = list(chain.from_iterable(x or [] for x in lang_df[col].tolist()))
        else:
            tokens = list(chain.from_iterable((str(x or "")).split() for x in lang_df[col].tolist()))
        tokens = [t.strip() for t in tokens if isinstance(t, str) and t.strip()]
        counts = pd.Series(tokens).value_counts().head(topn)

        out = counts.reset_index()
        out.columns = ["token", "count"]
        return out #


In [39]:
train_tr[train_tr["answerable"] == True]

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
0,4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,NaN,"['30년', '전쟁의', '승자는', '누구인가']",4,"['The', 'conflict', 'between', 'France', 'and'...",108,Who is the winner of the Thirty Years' War?
1,4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,NaN,"['엑스선은', '누가', '발견하였는가']",3,"['X-rays', 'make', 'up', 'X-radiation', 'a', '...",122,Who discovered X-rays?
2,4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,NaN,"['아테네에서', '언제', '가장', '최근의', '올림픽이', '올렸나요']",6,"['In', '2022', 'Beijing', 'will', 'become', 't...",197,When was the last Olympic Games held in Athens?
3,4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),NaN,"['세상에서', '가장', '오래된', '방송사는', '무엇인가']",5,"['The', 'British', 'Broadcasting', 'Corporatio...",70,What's the oldest broadcaster in the world?
4,4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,NaN,"['팔레스타인', '수도는', '어딘가요']",3,"['Palestine', '', '', 'officially', 'the', 'St...",84,Where's the Palestinian capital?
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6329,15337,서울교통공사는 언제 창단했나요?,Seoul Metro (Hangul: 서울메트로) is a public corpor...,ko,True,111,1970,NaN,"['서울교통공사는', '언제', '창단했나요']",3,"['Seoul', 'Metro', 'Hangul', '서울메트로', 'is', 'a...",47,When did you start Seoul Transit?
6330,15338,소말리아는 2차 개헌을 언제 했나요?,"In February 2012, Somali government officials ...",ko,True,923,23 June 2012,NaN,"['소말리아는', '2차', '개헌을', '언제', '했나요']",5,"['In', 'February', '2012', 'Somali', 'governme...",181,When did Somalia make its second inauguration?
6331,15339,세상에서 가장 먼저 시작된 교통수단은 무엇인가?,The first earth tracks were created by humans ...,ko,True,160,animals,NaN,"['세상에서', '가장', '먼저', '시작된', '교통수단은', '무엇인가']",6,"['The', 'first', 'earth', 'tracks', 'were', 'c...",118,What was the world's first transportation system?
6332,15340,2019년 이집트의 지도자는 누구인가?,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",ko,True,0,Abdel Fattah Saeed Hussein Khalil El-Sisi,NaN,"['2019년', '이집트의', '지도자는', '누구인가']",4,"['Abdel', 'Fattah', 'Saeed', 'Hussein', 'Khali...",30,Who is Egypt's leader in 2019?


In [40]:
train_tr[train_tr["answerable"] == False]

,Unnamed: 0,question,context,lang,answerable,answer_start,answer,answer_inlang,question_stripped,question_wordcount,context_stripped,context_wordcount,question_translated
17,4809,달은 자체발광하는가?,Moonlight consists of mostly sunlight (with li...,ko,False,-1,no,NaN,"['달은', '자체발광하는가']",2,"['Moonlight', 'consists', 'of', 'mostly', 'sun...",21,Does the moon glow by itself?
36,4828,포에니 전쟁에서 로마가 승리했나요?,With the end of the Macedonian Wars – which ra...,ko,False,-1,no,NaN,"['포에니', '전쟁에서', '로마가', '승리했나요']",4,"['With', 'the', 'end', 'of', 'the', 'Macedonia...",78,Was Rome victorious in the Phoenician War?
109,4901,악어는 땅과 물 둘 다에서 살 수 있을까?,"When on land, an American alligator moves eith...",ko,False,-1,no,NaN,"['악어는', '땅과', '물', '둘', '다에서', '살', '수', '있을까']",8,"['When', 'on', 'land', 'an', 'American', 'alli...",147,Can crocodiles live on both land and water?
115,4907,곰팡이가 상처를 치료할 수 있을까?,Many species produce metabolites that are majo...,ko,False,-1,no,NaN,"['곰팡이가', '상처를', '치료할', '수', '있을까']",5,"['Many', 'species', 'produce', 'metabolites', ...",301,Can mushrooms heal wounds?
119,4911,옥스퍼드 대학의 창립주는 누구인가요?,The University of Oxford has no known foundati...,ko,False,-1,no,NaN,"['옥스퍼드', '대학의', '창립주는', '누구인가요']",4,"['The', 'University', 'of', 'Oxford', 'has', '...",109,Who is the founder of Oxford University?
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6321,15322,క్షయ వ్యాధికి విరుగుడు ఏ దేశంలో కనుగొన్నారు?,Vaccines against anthrax for use in livestock ...,te,False,-1,France,ఫ్రాన్స్,"['క్షయ', 'వ్యాధికి', 'విరుగుడు', 'ఏ', 'దేశంలో'...",6,"['Vaccines', 'against', 'anthrax', 'for', 'use...",100,In what country was the antidote to tuberculos...
6322,15323,ఖురాన్ ఏ అరబ్బీ భాషలో ఎవరు రాసారు?,are broken Other Names of the Qur'an: It is be...,te,False,-1,Prophet Muhammad,ముహమ్మద్ ప్రవక్త,"['ఖురాన్', 'ఏ', 'అరబ్బీ', 'భాషలో', 'ఎవరు', 'రా...",6,"['are', 'broken', 'Other', 'Names', 'of', 'the...",139,Who wrote the Qur'an in which Arabic language?
6323,15324,టెక్సస్ రాష్ట్రంలోని అతిపెద్ద మానవ నిర్మితం ఏది ?,Austin is the capital of the US state of Texas...,te,False,-1,JP Morgan Chase Tower,జేపీ మోర్గాన్ ఛేజ్ టవర్,"['టెక్సస్', 'రాష్ట్రంలోని', 'అతిపెద్ద', 'మానవ'...",7,"['Austin', 'is', 'the', 'capital', 'of', 'the'...",135,What is the largest man-made structure in the ...
6324,15325,తమిళనాడులో రాష్ట్ర మొదటి ముఖ్యమంత్రి ఎవరు?,It became the 'Dravidakajagam' party under Nay...,te,False,-1,C. N. Annadurai,సి.ఎన్.అన్నాదురై,"['తమిళనాడులో', 'రాష్ట్ర', 'మొదటి', 'ముఖ్యమంత్ర...",5,"['It', 'became', 'the', 'Dravidakajagam', 'par...",143,Who was the first Chief Minister of Tamil Nadu?


### Idea: look at cosine difference between question and answer, but it will very likely fail. Other idea is too look at how questions are formed and if it is possible to extract certain information for the context. My theory that certain types of starting tockens correspond to non-answerable questions.

In [ ]:
train_set

In [ ]:
train_ds = load_dataset(
    "parquet",
    data_files={"train": "data/train.parquet"}
)["train"]

val_ds = load_dataset(
    "parquet",
    data_files={"validation": "data/validation.parquet"}
)["validation"]

test_ds = load_dataset(
    "json",
    data_files={"test": "data/test.json"}
)["test"]

In [ ]:
train_ds